# De Novo Drug Generation with DiffSBDD

Pocket-conditioned molecule generation using the **DiffSBDD** diffusion model.

| Cell | Purpose |
|------|---------|
| 1 — Config | Set your target protein and pocket parameters |
| 2 — Dependencies | Clone DiffSBDD, patch sources, install packages |
| 3 — Pocket Preparation | Download PDB and crop to binding site |
| 4 — Checkpoint | Fetch pre-trained weights from Zenodo |
| 5 — Trajectory Patch | Patch `generate_ligands.py` to save 4D denoising trajectories |
| 6 — Generation | Generate drug candidates |
| 7 — Validation | Quick inspection of generated molecules |
| 8 — RDKit Filter | Sanitize, score (QED/SA), and filter candidates |
| 9 — Payload | Package valid candidates + trajectories + pocket for download |

> **Before running:** set the runtime to GPU via **Runtime → Change runtime type → T4 GPU**.

## Cell 1 — Configuration

Set your target protein here. Everything downstream uses these variables.

To change targets, just update these values and re-run Cells 3–6.

In [1]:
# ── Cell 1: Configuration ─────────────────────────────────────────────────────
# Change these to target a different protein.

PDB_ID         = "3PTB"                      # RCSB PDB identifier
POCKET_CENTER  = (17.0, 14.0, 4.0)           # Active site center (Å)
POCKET_RADIUS  = 10.0                         # Radius around center (Å)
N_SAMPLES      = 10                           # Number of molecules to generate

# ── Derived paths (no need to edit) ───────────────────────────────────────────
ROOT       = "/content"
DIFFSBDD   = f"{ROOT}/DiffSBDD"
OB_ENV     = f"{ROOT}/ob_env"
RAW_PDB    = f"{ROOT}/{PDB_ID}.pdb"
POCKET_PDB = f"{ROOT}/{PDB_ID}_pocket.pdb"
CKPT_PATH  = f"{DIFFSBDD}/checkpoints/crossdocked_fullatom_cond.ckpt"
OUTPUT_DIR = f"{ROOT}/generated_drugs"
OUT_SDF    = f"{OUTPUT_DIR}/generated_mols.sdf"
TRAJ_DIR   = f"{OUTPUT_DIR}/trajectories"

# ── Filtering / packaging paths ───────────────────────────────────────────────
VALID_SDF       = f"{ROOT}/valid_candidates.sdf"
VALID_TRAJ_DIR  = f"{ROOT}/valid_trajectories"
PAYLOAD_ZIP     = f"{ROOT}/ensemble_payload.zip"

print(f"Target     : {PDB_ID}")
print(f"Pocket     : center={POCKET_CENTER}, radius={POCKET_RADIUS} Å")
print(f"Samples    : {N_SAMPLES}")
print(f"Pocket PDB : {POCKET_PDB}")
print(f"Output SDF : {OUT_SDF}")
print(f"Trajectories: {TRAJ_DIR}")

Target     : 3PTB
Pocket     : center=(17.0, 14.0, 4.0), radius=10.0 Å
Samples    : 10
Pocket PDB : /content/3PTB_pocket.pdb
Output SDF : /content/generated_drugs/generated_mols.sdf
Trajectories: /content/generated_drugs/trajectories


## Cell 2 — Dependency Installation

Clones DiffSBDD, patches source files for Python 3.12 / PyTorch 2.6+ compatibility,
and installs all required packages. Only needs to run **once per runtime session**.

In [2]:
# ── Cell 2: Dependency Installation ───────────────────────────────────────────
import os, sys, glob, subprocess, torch

# ── 2a. Clone DiffSBDD ────────────────────────────────────────────────────────
if not os.path.isdir(DIFFSBDD):
    !git clone https://github.com/arneschneuing/DiffSBDD.git {DIFFSBDD}
else:
    print(f"{DIFFSBDD}/ already present — skipping clone.")

# ── 2b. Patch DiffSBDD for modern dependencies ───────────────────────────────
#
#   Patch 1 — Biopython >= 1.82 removed three_to_one from Bio.PDB.Polypeptide
#   Patch 2 — PyTorch 2.6+ defaults torch.load(weights_only=True), but the
#             checkpoint serialises argparse.Namespace which the safe unpickler
#             rejects. pytorch_lightning explicitly passes weights_only=True,
#             so we must force it to False.

# --- Patch 1: three_to_one shim ---
lm = f"{DIFFSBDD}/lightning_modules.py"
with open(lm) as f:
    src = f.read()
if '[PATCHED]' not in src:
    src = src.replace(
        'from Bio.PDB.Polypeptide import three_to_one',
        '# [PATCHED] Biopython >= 1.82 removed three_to_one\n'
        'try:\n'
        '    from Bio.PDB.Polypeptide import three_to_one\n'
        'except ImportError:\n'
        '    from Bio.Data.IUPACData import protein_letters_3to1\n'
        '    def three_to_one(s): return protein_letters_3to1[s.upper().strip()]'
    )
    with open(lm, 'w') as f:
        f.write(src)
    print("✓ Patched lightning_modules.py — three_to_one shim")
else:
    print("✓ lightning_modules.py — already patched")

# --- Patch 2: force weights_only=False ---
gl = f"{DIFFSBDD}/generate_ligands.py"
with open(gl) as f:
    src = f.read()
if '_patched_load' not in src:
    patch = (
        '# [PATCHED] PyTorch 2.6+ / pytorch_lightning passes weights_only=True\n'
        'import torch as _torch\n'
        '_orig_torch_load = _torch.load\n'
        'def _patched_load(*a, **kw):\n'
        '    kw["weights_only"] = False\n'
        '    return _orig_torch_load(*a, **kw)\n'
        '_torch.load = _patched_load\n'
    )
    lines = src.split('\n', 1)
    src = lines[0] + '\n' + patch + lines[1]
    with open(gl, 'w') as f:
        f.write(src)
    print("✓ Patched generate_ligands.py — torch.load weights_only=False")
else:
    print("✓ generate_ligands.py — already patched")

# ── 2c. Detect PyTorch + CUDA versions ───────────────────────────────────────
torch_base    = torch.__version__.split('+')[0]
cuda_str      = torch.version.cuda
cuda_short    = 'cu' + cuda_str.replace('.', '')[:3]
pyg_wheel_url = f'https://data.pyg.org/whl/torch-{torch_base}+{cuda_short}.html'

print(f"\nRuntime: PyTorch {torch.__version__} | CUDA {cuda_str}")
print(f"PyG wheels: {pyg_wheel_url}\n")

# ── 2d. Purge stale PyG extensions ────────────────────────────────────────────
!pip uninstall -y torch_scatter torch_sparse torch_cluster torch_spline_conv pyg_lib 2>/dev/null || true

# ── 2e. OpenBabel via micromamba ───────────────────────────────────────────────
MAMBA  = '/usr/local/bin/micromamba'
py_ver = f'{sys.version_info.major}.{sys.version_info.minor}'

if not os.path.isfile(MAMBA):
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /usr/local bin/micromamba 2>/dev/null

if not os.path.isdir(OB_ENV):
    print(f"Creating OpenBabel environment (python={py_ver})…")
    !{MAMBA} create -y -p {OB_ENV} -c conda-forge openbabel python={py_ver} --quiet
else:
    print(f"OpenBabel env exists at {OB_ENV}")

ob_site = glob.glob(f'{OB_ENV}/lib/python{py_ver}/site-packages')
ob_site = ob_site[0] if ob_site else ''
if ob_site and ob_site not in sys.path:
    sys.path.insert(0, ob_site)
os.environ['LD_LIBRARY_PATH'] = f"{OB_ENV}/lib:{os.environ.get('LD_LIBRARY_PATH', '')}"

import ctypes
ctypes.CDLL(f'{OB_ENV}/lib/libopenbabel.so')
from openbabel import openbabel as _ob_test
print(f"✓ OpenBabel loaded")

# ── 2f. All DiffSBDD deps (single pip call with numpy constraint) ─────────────
!pip install -q \
    "numpy>=1.26,<2.1" \
    pytorch-lightning wandb biopython imageio scipy \
    networkx pandas seaborn torchmetrics tqdm protobuf rdkit

# ── 2g. PyTorch Geometric + extensions ────────────────────────────────────────
!pip install -q torch_geometric
!pip install -q --force-reinstall --no-cache-dir \
    torch_scatter torch_sparse torch_cluster torch_spline_conv \
    -f {pyg_wheel_url}
!pip install -q --no-cache-dir pyg_lib -f {pyg_wheel_url} \
    || echo "pyg_lib not available — skipping (non-critical)."

# ── Sanity check (subprocess — safe across numpy version changes) ─────────────
sanity = subprocess.run(
    ['python', '-c', '\n'.join([
        "import numpy; print(f'  numpy             : {numpy.__version__}')",
        "import torch; print(f'  PyTorch           : {torch.__version__}')",
        "import torch_scatter; print(f'  torch_scatter     : {torch_scatter.__version__}')",
        "import pytorch_lightning as pl; print(f'  pytorch_lightning : {pl.__version__}')",
        "from Bio.PDB.Polypeptide import is_aa; print('  Biopython         : ✓')",
        "from openbabel import openbabel; print('  OpenBabel         : ✓')",
        "from rdkit import Chem; print('  RDKit             : ✓')",
        "import scipy; print(f'  scipy             : {scipy.__version__}')",
    ])],
    env={**os.environ,
         'PYTHONPATH': f"{ob_site}:{os.environ.get('PYTHONPATH', '')}",
         'LD_LIBRARY_PATH': f"{OB_ENV}/lib:{os.environ.get('LD_LIBRARY_PATH', '')}"},
    capture_output=True, text=True
)

if sanity.returncode == 0:
    print(f"\n✅ All dependencies OK:")
    print(sanity.stdout.rstrip())
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE'
    print(f"  GPU               : {gpu}")
else:
    print(f"\n❌ Sanity check failed:\n{sanity.stdout}\n{sanity.stderr}")

Cloning into '/content/DiffSBDD'...
remote: Enumerating objects: 238, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 238 (delta 103), reused 73 (delta 73), pack-reused 87 (from 2)
Receiving objects: 100% (238/238), 107.89 MiB | 31.48 MiB/s, done.
Resolving deltas: 100% (127/127), done.
✓ Patched lightning_modules.py — three_to_one shim
✓ Patched generate_ligands.py — torch.load weights_only=False

Runtime: PyTorch 2.10.0+cu128 | CUDA 12.8
PyG wheels: https://data.pyg.org/whl/torch-2.10.0+cu128.html

bin/micromamba
Creating OpenBabel environment (python=3.12)…
✓ OpenBabel loaded
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 90.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 20.7 MB/s eta 0:00:00:00:0100:01
     ━━━━━

## Interlude — ConforMix Installation

Clones and installs the ConforMix package for induced-fit structural variation via the Boltz backend.

In [13]:
!git clone https://github.com/drorlab/conformix.git
!pip install ./conformix/conformix_boltz

fatal: destination path 'conformix' already exists and is not an empty directory.
Processing ./conformix/conformix_boltz
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached hydra_core-1.3.2-py3-none-any.whl.metadata (5.5 kB)
  Using cached pytorch_lightning-2.4.0-py3-none-any.whl.metadata (21 kB)
  Using cached dm_tree-0.1.9-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.4 kB)
  Using cached types_requests-2.33.0.20260408-py3-none-any.whl.metadata (2.0 kB)
  Using cached einops-0.8.1-py3-none-any.whl.metadata (13 kB)
  Using cached einx-0.3.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached fairscale-0.4.13.tar.gz (266 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached mashumaro-3.14-py3-none-any.whl.metadata (114 kB)
  U

## Cell 3 — Pocket Preparation

Downloads the full PDB structure from RCSB and crops it to the binding pocket
defined by `POCKET_CENTER` and `POCKET_RADIUS` from Cell 1.

**To use a custom pocket PDB:** upload it to `/content/`, update `POCKET_PDB`
in Cell 1, and skip this cell.

In [4]:
# ── Cell 3: Pocket Preparation ────────────────────────────────────────────────
import os, subprocess, urllib.request

url_cif = f"https://files.rcsb.org/download/{PDB_ID}.cif"
url_fasta = f"https://www.rcsb.org/fasta/entry/{PDB_ID}/display"
cif_path = f"/content/{PDB_ID}.cif"
fasta_path = f"/content/{PDB_ID}.fasta"

if not os.path.isfile(cif_path):
    print(f"Downloading {PDB_ID} CIF...")
    urllib.request.urlretrieve(url_cif, cif_path)
if not os.path.isfile(fasta_path):
    print(f"Downloading {PDB_ID} FASTA...")
    urllib.request.urlretrieve(url_fasta, fasta_path)
    # Reformat RCSB FASTA to Boltz standard (>ChainID|protein)
    with open(fasta_path, "r") as f:
        flines = f.readlines()
    with open(fasta_path, "w") as f:
        for fline in flines:
            if fline.startswith(">"):
                chain = "A"
                parts = fline.split("|")
                if len(parts) > 1 and "Chain" in parts[1]:
                    chain = parts[1].replace("Chain ", "").strip()
                    if "," in chain: chain = chain.split(",")[0] # handle multiple chains
                f.write(f">{chain}|protein\n")
            else:
                f.write(fline)

if os.path.isfile(POCKET_PDB):
    size_kb = os.path.getsize(POCKET_PDB) / 1_000
    print(f"✅ Pocket PDB already exists: {POCKET_PDB} ({size_kb:.1f} KB)")
else:
    cx, cy, cz = POCKET_CENTER
    script = f'''
import numpy as np
from Bio.PDB import PDBParser, PDBIO, Select
import urllib.request, os

pdb_id     = "{PDB_ID}"
raw_pdb    = "{RAW_PDB}"
pocket_pdb = "{POCKET_PDB}"
center     = np.array([{cx}, {cy}, {cz}])
radius     = {POCKET_RADIUS}

print(f"Downloading {{pdb_id}} from RCSB PDB…")
urllib.request.urlretrieve(
    f"https://files.rcsb.org/download/{{pdb_id}}.pdb", raw_pdb)

structure = PDBParser(QUIET=True).get_structure(pdb_id, raw_pdb)

class PocketSelect(Select):
    def accept_residue(self, residue):
        for atom in residue:
            if np.linalg.norm(atom.get_vector().get_array() - center) < radius:
                return True
        return False

io = PDBIO()
io.set_structure(structure)
io.save(pocket_pdb, PocketSelect())

size_kb = os.path.getsize(pocket_pdb) / 1_000
print(f"✅ Pocket saved: {{pocket_pdb}} ({{size_kb:.1f}} KB)")
print(f"   Center: ({cx}, {cy}, {cz}), Radius: {POCKET_RADIUS} Å")
'''
    result = subprocess.run(['python', '-c', script],
                           capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(f"❌ Pocket extraction failed:\n{result.stderr}")
    else:
        assert os.path.isfile(POCKET_PDB), f"Pocket file not created at {POCKET_PDB}"

✅ Pocket saved: /content/3PTB_pocket.pdb (4.9 KB)
   Center: (17.0, 14.0, 4.0), Radius: 10.0 Å



## Cell 4 — Checkpoint Download

Downloads `crossdocked_fullatom_cond.ckpt` (~17 MB) from Zenodo.

In [5]:
# ── Cell 4: Checkpoint Download ───────────────────────────────────────────────
import os

CKPT_FILE = "crossdocked_fullatom_cond.ckpt"
CKPT_DIR  = f"{DIFFSBDD}/checkpoints"
CKPT_URL  = f"https://zenodo.org/records/8183747/files/{CKPT_FILE}?download=1"

if os.path.isfile(CKPT_PATH):
    size_mb = os.path.getsize(CKPT_PATH) / 1_000_000
    if size_mb < 1:
        print(f"Corrupt file ({size_mb:.2f} MB) — re-downloading.")
        os.remove(CKPT_PATH)

if not os.path.isfile(CKPT_PATH):
    os.makedirs(CKPT_DIR, exist_ok=True)
    print(f"Downloading {CKPT_FILE} from Zenodo…")
    !wget -q --show-progress -O "{CKPT_PATH}" "{CKPT_URL}"
    size_mb = os.path.getsize(CKPT_PATH) / 1_000_000
    assert size_mb > 1, f"File is only {size_mb:.2f} MB — likely corrupt."
    print(f"\n✅ Checkpoint: {CKPT_PATH} ({size_mb:.1f} MB)")
else:
    size_mb = os.path.getsize(CKPT_PATH) / 1_000_000
    print(f"✅ Checkpoint already present: {CKPT_PATH} ({size_mb:.1f} MB)")

/content/DiffSBDD/c 100%[===================>]  17.03M  17.1MB/s    in 1.0s    

✅ Checkpoint: /content/DiffSBDD/checkpoints/crossdocked_fullatom_cond.ckpt (17.9 MB)


## Cell 5 — Trajectory Patch

Patches `generate_ligands.py` to capture **4D denoising trajectories**.

The patch monkey-patches `LigandPocketDDPM.generate_ligands` so that the
intermediate 3D coordinates at each denoising step are recorded. For every
generated molecule, a multi-frame `.xyz` file is saved to
`generated_drugs/trajectories/mol_<i>.xyz`.

**Run this cell _before_ Cell 6 (Generation).**

In [6]:
# ── Cell 5: Trajectory Patch ──────────────────────────────────────────────────
#
# Patches generate_ligands.py to save intermediate denoising snapshots.
#
# Strategy: we modify the script so that model.generate_ligands() also returns
# trajectory frames by passing return_frames to the underlying DDPM sampler,
# then writes per-molecule multi-frame .xyz files before the normal SDF output.
#
# The patch is idempotent — safe to re-run.

import os, textwrap

gl = f"{DIFFSBDD}/generate_ligands.py"
with open(gl) as f:
    src = f.read()

TRAJ_MARKER = '# [TRAJ_PATCH]'

if TRAJ_MARKER not in src:
    # ── Build the trajectory-saving patch ─────────────────────────────────────
    #
    # We inject code that:
    #   1. Monkey-patches the DDPM's sample_given_pocket to capture all
    #      intermediate frames (return_frames = timesteps).
    #   2. After generation, writes per-molecule multi-frame .xyz files.
    #
    # NOTE: We use regular strings (NOT f-strings) for the injected code blocks
    # to avoid curly-brace conflicts with Python dict literals and format specs
    # in the generated code.

    traj_block = (
        '\n'
        '    ' + TRAJ_MARKER + '\n'
        '    # ── Save 4D denoising trajectories ────────────────────────────────\n'
        '    import sys as _sys\n'
        '    _sys.path.insert(0, "' + DIFFSBDD + '")\n'
        '    from constants import dataset_params\n'
        '\n'
        '    _traj_dir = str(Path(args.outfile).parent / "trajectories")\n'
        '    Path(_traj_dir).mkdir(parents=True, exist_ok=True)\n'
        '\n'
        '    _trajectory_frames = []\n'
        '\n'
        '    # Atom decoder for element symbols\n'
        '    _atom_decoder = model.dataset_info["atom_decoder"]\n'
        '\n'
        '    # Monkey-patch sample_given_pocket to record intermediate coords\n'
        '    _orig_sample_gp = model.ddpm.sample_given_pocket.__func__ if hasattr(\n'
        '        model.ddpm.sample_given_pocket, "__func__") else None\n'
        '\n'
        '    if _orig_sample_gp is not None:\n'
        '        import torch as _t\n'
        '        from torch_scatter import scatter_mean as _scatter_mean\n'
        '\n'
        '        def _patched_sample_given_pocket(self, pocket, num_nodes_lig,\n'
        '                                         return_frames=1, timesteps=None):\n'
        '            _ts = self.T if timesteps is None else timesteps\n'
        '            out_lig, out_pocket, lig_mask, pocket_mask = \\\n'
        '                _orig_sample_gp(self, pocket, num_nodes_lig,\n'
        '                                return_frames=_ts, timesteps=timesteps)\n'
        '            _trajectory_frames.append(dict(\n'
        '                out_lig=out_lig.detach().cpu(),\n'
        '                lig_mask=lig_mask.detach().cpu(),\n'
        '            ))\n'
        '            # Return only the final frame (frame 0)\n'
        '            _out_pocket = out_pocket[0] if out_pocket.dim() > 2 else out_pocket\n'
        '            return out_lig[0], _out_pocket, lig_mask, pocket_mask\n'
        '\n'
        '        model.ddpm.sample_given_pocket = \\\n'
        '            lambda *a, **kw: _patched_sample_given_pocket(model.ddpm, *a, **kw)\n'
        '        print("✓ Trajectory capture enabled (saving to " + _traj_dir + "/)")\n'
        '\n'
    )

    # Insert the trajectory block BEFORE the molecules = [] line
    src = src.replace(
        '    molecules = []\n    for i in range(args.n_samples // args.batch_size):',
        traj_block + '    molecules = []\n    for i in range(args.n_samples // args.batch_size):'
    )

    # Now add the trajectory-saving code AFTER the SDF write
    traj_save_block = (
        '\n'
        '    # ── Write trajectory .xyz files ───────────────────────────────────\n'
        '    if _trajectory_frames:\n'
        '        import numpy as _np\n'
        '        print("\\nSaving denoising trajectories…")\n'
        '        mol_idx = 0\n'
        '        for batch_data in _trajectory_frames:\n'
        '            out_lig = batch_data["out_lig"]\n'
        '            lig_mask = batch_data["lig_mask"]\n'
        '            n_frames = out_lig.shape[0]\n'
        '\n'
        '            unique_mols = lig_mask.unique()\n'
        '            for m_id in unique_mols:\n'
        '                atom_indices = (lig_mask == m_id).nonzero(as_tuple=True)[0]\n'
        '                n_atoms = len(atom_indices)\n'
        '\n'
        '                xyz_path = _traj_dir + "/mol_" + str(mol_idx).zfill(4) + ".xyz"\n'
        '                with open(xyz_path, "w") as fout:\n'
        '                    for frame_i in reversed(range(n_frames)):\n'
        '                        frame = out_lig[frame_i]\n'
        '                        coords = frame[atom_indices, :3].numpy()\n'
        '                        types  = frame[atom_indices, 3:].argmax(dim=1).numpy()\n'
        '\n'
        '                        fout.write(str(n_atoms) + "\\n")\n'
        '                        fout.write("Frame " + str(n_frames - 1 - frame_i)\n'
        '                                   + " / " + str(n_frames - 1)\n'
        '                                   + " | mol_" + str(mol_idx) + "\\n")\n'
        '                        for ai in range(n_atoms):\n'
        '                            elem = _atom_decoder[types[ai]] if types[ai] < len(_atom_decoder) else "X"\n'
        '                            fout.write("%-2s %10.4f %10.4f %10.4f\\n" % (\n'
        '                                elem, coords[ai,0], coords[ai,1], coords[ai,2]))\n'
        '                mol_idx += 1\n'
        '\n'
        '        print("✅ Saved " + str(mol_idx) + " trajectory files to " + _traj_dir + "/")\n'
    )

    src = src.replace(
        '    # Make SDF files\n    utils.write_sdf_file(args.outfile, molecules)',
        '    # Make SDF files\n    utils.write_sdf_file(args.outfile, molecules)\n'
        + traj_save_block
    )

    with open(gl, 'w') as f:
        f.write(src)
    print(f"✓ Patched generate_ligands.py — 4D trajectory capture")
else:
    print(f"✓ generate_ligands.py — trajectory patch already applied")

# Create trajectories directory
os.makedirs(TRAJ_DIR, exist_ok=True)
print(f"\n📁 Trajectory directory: {TRAJ_DIR}")


✓ Patched generate_ligands.py — 4D trajectory capture

📁 Trajectory directory: /content/generated_drugs/trajectories


## Cell 6 — De Novo Drug Generation

Extracts residue IDs from the pocket PDB and runs `generate_ligands.py`.

With the trajectory patch applied (Cell 5), intermediate denoising snapshots
will also be saved to `generated_drugs/trajectories/`.

In [7]:
# ── Cell 6: De Novo Drug Generation ───────────────────────────────────────────
import os, sys, subprocess, glob

assert os.path.isfile(CKPT_PATH),  f"Checkpoint missing — re-run Cell 4. ({CKPT_PATH})"
assert os.path.isfile(POCKET_PDB), f"Pocket PDB missing — re-run Cell 3. ({POCKET_PDB})"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Extract residue IDs from pocket PDB ───────────────────────────────────────
res = subprocess.run(['python', '-c', f'''
from Bio.PDB import PDBParser
s = PDBParser(QUIET=True).get_structure("p", "{POCKET_PDB}")
resi = [f"{{c.id}}:{{r.id[1]}}" for c in s[0] for r in c if r.id[0] == " "]
print(" ".join(resi))
'''], capture_output=True, text=True)

resi_list = res.stdout.strip()
n_res = len(resi_list.split())
assert n_res > 0, f"No residues found in {POCKET_PDB}"

print(f"Configuration:")
print(f"  Target     : {PDB_ID}")
print(f"  Pocket     : {POCKET_PDB} ({n_res} residues)")
print(f"  Checkpoint : {CKPT_PATH}")
print(f"  Samples    : {N_SAMPLES}")
print(f"  Output     : {OUT_SDF}")
print(f"\nLaunching DiffSBDD…\n")

# ── Build environment with OpenBabel paths ────────────────────────────────────
py_ver  = f'{sys.version_info.major}.{sys.version_info.minor}'
ob_site = glob.glob(f'{OB_ENV}/lib/python{py_ver}/site-packages')
ob_site = ob_site[0] if ob_site else ''

cmd = (
    f'PYTHONPATH="{ob_site}:$PYTHONPATH" '
    f'LD_LIBRARY_PATH="{OB_ENV}/lib:$LD_LIBRARY_PATH" '
    f'python {DIFFSBDD}/generate_ligands.py '
    f'{CKPT_PATH} '
    f'--pdbfile {POCKET_PDB} '
    f'--resi_list {resi_list} '
    f'--n_samples {N_SAMPLES} '
    f'--outfile {OUT_SDF}'
)

ret = os.system(cmd)

if ret != 0:
    print(f"\n❌ Generation failed (exit code {ret}).")
else:
    print("\n" + "═" * 60)
    print("✅ Generation complete!")
    print("═" * 60)
    if os.path.isfile(OUT_SDF):
        size_kb = os.path.getsize(OUT_SDF) / 1_000
        print(f"  Output: {OUT_SDF} ({size_kb:.1f} KB)")
    else:
        print(f"  ⚠ Output file not found at {OUT_SDF}")
    # Report trajectory files
    traj_files = glob.glob(f"{TRAJ_DIR}/mol_*.xyz")
    if traj_files:
        print(f"  Trajectories: {len(traj_files)} files in {TRAJ_DIR}/")
    else:
        print(f"  ⚠ No trajectory files found in {TRAJ_DIR}/")

Configuration:
  Target     : 3PTB
  Pocket     : /content/3PTB_pocket.pdb (8 residues)
  Checkpoint : /content/DiffSBDD/checkpoints/crossdocked_fullatom_cond.ckpt
  Samples    : 10
  Output     : /content/generated_drugs/generated_mols.sdf

Launching DiffSBDD…


════════════════════════════════════════════════════════════
✅ Generation complete!
════════════════════════════════════════════════════════════
  Output: /content/generated_drugs/generated_mols.sdf (9.7 KB)
  Trajectories: 10 files in /content/generated_drugs/trajectories/


## Cell 7 — Validation

Quick inspection of the generated molecules.

In [8]:
# ── Cell 7: Validation ────────────────────────────────────────────────────────
import os, subprocess

assert os.path.isfile(OUT_SDF), f"SDF not found at {OUT_SDF} — run Cell 6 first."

res = subprocess.run(['python', '-c', f'''
from rdkit import Chem
from rdkit.Chem import Descriptors
import os

sdf = "{OUT_SDF}"
size_kb = os.path.getsize(sdf) / 1000
suppl = list(Chem.SDMolSupplier(sdf))
valid = [m for m in suppl if m is not None]

print(f"File       : {{sdf}}")
print(f"Size       : {{size_kb:.1f}} KB")
print(f"Total      : {{len(suppl)}}")
print(f"Valid      : {{len(valid)}} ({{100*len(valid)/max(len(suppl),1):.0f}}%)")

if valid:
    mws = [Descriptors.MolWt(m) for m in valid]
    has = [m.GetNumHeavyAtoms() for m in valid]
    print(f"MW range   : {{min(mws):.0f}} – {{max(mws):.0f}} Da")
    print(f"Atoms range: {{min(has)}} – {{max(has)}} heavy atoms")
    print(f"\\nSample molecules:")
    for i, m in enumerate(valid[:5]):
        print(f"  {{i+1}}. MW={{Descriptors.MolWt(m):.0f}}, "
              f"atoms={{m.GetNumHeavyAtoms()}}, "
              f"SMILES={{Chem.MolToSmiles(m)}}")
'''], capture_output=True, text=True)

print(res.stdout)
if res.stderr:
    print(res.stderr)

File       : /content/generated_drugs/generated_mols.sdf
Size       : 9.7 KB
Total      : 10
Valid      : 9 (90%)
MW range   : 115 – 216 Da
Atoms range: 8 – 14 heavy atoms

Sample molecules:
  1. MW=173, atoms=11, SMILES=C[C@@H]1[C@H]2CNC[S@@H]1[C@H]1[C@@H](O)[C@@H]21
  2. MW=162, atoms=11, SMILES=O=C1[C@H](O)O[C@H](CO)[C@H]1CO
  3. MW=126, atoms=9, SMILES=OC1(C2=NCCN2)CC1
  4. MW=216, atoms=14, SMILES=CC=C=CC=CCCCCP(=O)(O)O
  5. MW=133, atoms=8, SMILES=C=C(CC)N(S)CO

[07:28:06] Explicit valence for atom # 2 C, 5, is greater than permitted
[07:28:06] ERROR: Could not sanitize molecule ending on line 216
[07:28:06] ERROR: Explicit valence for atom # 2 C, 5, is greater than permitted



## Cell 8 — RDKit Chemistry Filter

Iterates through the generated `.sdf` molecules and applies a chemistry filter:

1. **Sanitization** — Drop molecules that fail RDKit sanitization (invalid valence/bonds).
2. **QED** (Quantitative Estimate of Drug-likeness) — Drop if QED < 0.3.
3. **SA** (Synthetic Accessibility) — Drop if SA score > 6.0.

Passing molecules are saved as `valid_candidates.sdf`, and only their
corresponding trajectory files are copied to `valid_trajectories/`.

In [9]:
# ── Cell 8: RDKit Chemistry Filter ────────────────────────────────────────────
import os, sys, shutil, glob

assert os.path.isfile(OUT_SDF), f"SDF not found at {OUT_SDF} — run Cell 6 first."

from rdkit import Chem
from rdkit.Chem import QED as QED_module
from rdkit.Chem import Descriptors

# ── Import SA scorer from RDKit Contrib ────────────────────────────────────────
# The SA scorer lives in RDKit's Contrib directory and is not a standard import.
import rdkit
_rdkit_root = os.path.dirname(os.path.dirname(rdkit.__file__))
_sa_paths = [
    os.path.join(_rdkit_root, 'Contrib', 'SA_Score'),
    os.path.join(_rdkit_root, 'rdkit', 'Contrib', 'SA_Score'),
    # Common Colab / pip install path
    os.path.join(os.path.dirname(rdkit.__file__), 'Contrib', 'SA_Score'),
]
# Also check site-packages siblings
_site = os.path.dirname(_rdkit_root)
_sa_paths.append(os.path.join(_site, 'Contrib', 'SA_Score'))
_sa_paths.append(os.path.join(_site, 'rdkit', 'Contrib', 'SA_Score'))

_sa_found = False
for _p in _sa_paths:
    if os.path.isdir(_p):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        _sa_found = True
        print(f"✓ SA scorer found at: {_p}")
        break

if not _sa_found:
    # Fallback: try to find it anywhere under the rdkit installation
    _search_root = os.path.dirname(rdkit.__file__)
    for root, dirs, files in os.walk(os.path.dirname(_search_root)):
        if 'sascorer.py' in files:
            if root not in sys.path:
                sys.path.insert(0, root)
            _sa_found = True
            print(f"✓ SA scorer found at: {root}")
            break

if not _sa_found:
    print("⚠ SA scorer not found — will skip SA filtering.")
    print("  Looked in:", _sa_paths)

try:
    import sascorer
    _has_sa = True
    print("✓ sascorer imported successfully")
except ImportError:
    _has_sa = False
    print("⚠ Could not import sascorer — SA filtering disabled.")

# ── Thresholds ─────────────────────────────────────────────────────────────────
QED_MIN = 0.3
SA_MAX  = 6.0

# ── Load and filter molecules ─────────────────────────────────────────────────
suppl = list(Chem.SDMolSupplier(OUT_SDF))
print(f"\nLoaded {len(suppl)} molecules from {OUT_SDF}")
print(f"Thresholds: QED ≥ {QED_MIN}, SA ≤ {SA_MAX}\n")

valid_mols = []
valid_indices = []   # original indices of passing molecules
stats = {'total': len(suppl), 'parse_fail': 0, 'sanitize_fail': 0,
         'qed_fail': 0, 'sa_fail': 0, 'pass': 0}

for i, mol in enumerate(suppl):
    # 1. Parse check
    if mol is None:
        stats['parse_fail'] += 1
        continue

    # 2. Sanitize
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        stats['sanitize_fail'] += 1
        continue

    # 3. QED
    qed_score = QED_module.qed(mol)
    if qed_score < QED_MIN:
        stats['qed_fail'] += 1
        continue

    # 4. SA score
    if _has_sa:
        sa_score = sascorer.calculateScore(mol)
        if sa_score > SA_MAX:
            stats['sa_fail'] += 1
            continue
    else:
        sa_score = None

    # ── Passed all filters ────────────────────────────────────────────────────
    mol.SetProp('QED', f'{qed_score:.3f}')
    if sa_score is not None:
        mol.SetProp('SA_Score', f'{sa_score:.3f}')
    mol.SetProp('OriginalIndex', str(i))
    valid_mols.append(mol)
    valid_indices.append(i)
    stats['pass'] += 1

# ── Report ────────────────────────────────────────────────────────────────────
print("═" * 50)
print(f"  Total molecules     : {stats['total']}")
print(f"  Parse failures      : {stats['parse_fail']}")
print(f"  Sanitization fails  : {stats['sanitize_fail']}")
print(f"  QED < {QED_MIN} rejected  : {stats['qed_fail']}")
if _has_sa:
    print(f"  SA > {SA_MAX} rejected  : {stats['sa_fail']}")
print(f"  ✅ Passed           : {stats['pass']}")
print("═" * 50)

# ── Save valid candidates ─────────────────────────────────────────────────────
if valid_mols:
    writer = Chem.SDWriter(VALID_SDF)
    for mol in valid_mols:
        writer.write(mol)
    writer.close()
    size_kb = os.path.getsize(VALID_SDF) / 1_000
    print(f"\n✅ Valid candidates saved: {VALID_SDF} ({size_kb:.1f} KB)")

    # ── Show top candidates ────────────────────────────────────────────────────
    print(f"\nTop candidates:")
    for j, mol in enumerate(valid_mols[:5]):
        qed_val = mol.GetProp('QED')
        sa_val = mol.GetProp('SA_Score') if mol.HasProp('SA_Score') else 'N/A'
        smi = Chem.MolToSmiles(mol)
        print(f"  {j+1}. QED={qed_val}, SA={sa_val}, "
              f"MW={Descriptors.MolWt(mol):.0f}, SMILES={smi}")
else:
    print("\n⚠ No molecules passed the filters.")

# ── Move valid trajectories ───────────────────────────────────────────────────
if os.path.isdir(TRAJ_DIR):
    os.makedirs(VALID_TRAJ_DIR, exist_ok=True)
    moved = 0
    for idx in valid_indices:
        src_xyz = os.path.join(TRAJ_DIR, f"mol_{idx:04d}.xyz")
        if os.path.isfile(src_xyz):
            dst_xyz = os.path.join(VALID_TRAJ_DIR, f"mol_{idx:04d}.xyz")
            shutil.copy2(src_xyz, dst_xyz)
            moved += 1
    print(f"\n📁 Copied {moved} trajectory files to {VALID_TRAJ_DIR}/")
else:
    print(f"\n⚠ Trajectory directory not found at {TRAJ_DIR} — skipping.")

✓ SA scorer found at: /usr/local/lib/python3.12/dist-packages/rdkit/Contrib/SA_Score
✓ sascorer imported successfully

Loaded 10 molecules from /content/generated_drugs/generated_mols.sdf
Thresholds: QED ≥ 0.3, SA ≤ 6.0



[07:28:06] Explicit valence for atom # 2 C, 5, is greater than permitted
[07:28:06] ERROR: Could not sanitize molecule ending on line 216
[07:28:06] ERROR: Explicit valence for atom # 2 C, 5, is greater than permitted


══════════════════════════════════════════════════
  Total molecules     : 10
  Parse failures      : 1
  Sanitization fails  : 0
  QED < 0.3 rejected  : 0
  SA > 6.0 rejected  : 1
  ✅ Passed           : 8
══════════════════════════════════════════════════

✅ Valid candidates saved: /content/valid_candidates.sdf (8.1 KB)

Top candidates:
  1. QED=0.432, SA=4.298, MW=162, SMILES=O=C1[C@H](O)O[C@H](CO)[C@H]1CO
  2. QED=0.498, SA=3.602, MW=126, SMILES=OC1(C2=NCCN2)CC1
  3. QED=0.310, SA=3.704, MW=216, SMILES=CC=C=CC=CCCCCP(=O)(O)O
  4. QED=0.443, SA=4.250, MW=133, SMILES=C=C(CC)N(S)CO
  5. QED=0.498, SA=2.736, MW=115, SMILES=O=C(O)[C@H]1CCCN1

📁 Copied 8 trajectory files to /content/valid_trajectories/


## Interlude — Generate Ensemble using ConforMix

Run ConforMix to generate target structural variations locked to the CIF reference coordinates.

In [14]:
!sed -i "s|res_num = int(atom.resi)|res_num = int(''.join(filter(str.isdigit, str(atom.resi))))|g" /usr/local/lib/python3.12/dist-packages/boltz/run_twisted.py
!sed -i 's/use_msa_server: bool = False/use_msa_server: bool = True/g' /usr/local/lib/python3.12/dist-packages/boltz/run_twisted.py
!sed -i 's/use_msa_server=False/use_msa_server=True/g' /usr/local/lib/python3.12/dist-packages/boltz/run_twisted.py
!python -m boltz.run_conformixrmsd_boltz \
    --fasta-path /content/3PTB.fasta \
    --reference-cif /content/3PTB.cif \
    --subset-residues "40-60" \
    --out-dir /content/conformix_outputs \
    --twist-target-start 0.0 \
    --twist-target-stop 2.0 \
    --num-twist-targets 3 \
    --samples-per-target 2 \
    --structured-regions-only


--- Settings ---
fasta_path               : /content/3PTB.fasta
out_dir                  : /content/conformix_outputs
structured_regions_only  : True
subset_residues          : 40-60
reference_cif            : /content/3PTB.cif
twist_target_start       : 0.0
twist_target_stop        : 2.0
num_twist_targets        : 3
samples_per_target       : 2
twist_strength           : 15.0
tstart                   : 200
tstop                    : 0
accelerator              : gpu
devices                  : 1
cache                    : ~/.cache
----------------

Loading Boltz model...
Model loaded successfully.

Using user-provided reference: /content/3PTB.cif
INFO: Guiding RMSD on structured regions (alpha-helices and beta-sheets).

Starting guided sampling with 3 target values...
Target RMSDs (Å): ['0.00', '1.00', '2.00']
Checking input data.
Running predictions for 1 structure
Processing input data.
  0% 0/1 [00:00<?, ?it/s]Generating MSA for /content/3PTB.fasta with 1 protein entities.

  0% 0/15

## Cell 9 — Payload Packaging

Bundles the final deliverables into `ensemble_payload.zip` and triggers a
download to your local machine:

- `valid_candidates.sdf` — Filtered drug candidates with QED/SA annotations
- `valid_trajectories/` — 4D denoising trajectory `.xyz` files for valid molecules
- `pocket.pdb` — The target protein binding pocket

In [15]:
# ── Cell 9: Payload Packaging ─────────────────────────────────────────────────
import os, shutil, glob, json

assert os.path.isfile(VALID_SDF), \
    f"valid_candidates.sdf not found at {VALID_SDF} — run Cell 8 first."
assert os.path.isfile(POCKET_PDB), \
    f"Pocket PDB not found at {POCKET_PDB} — run Cell 3 first."

# ── Stage files for archiving ─────────────────────────────────────────────────
STAGING_DIR = f"{ROOT}/ensemble_payload_staging"
if os.path.isdir(STAGING_DIR):
    shutil.rmtree(STAGING_DIR)
os.makedirs(STAGING_DIR, exist_ok=True)

# 1. valid_candidates.sdf
shutil.copy2(VALID_SDF, os.path.join(STAGING_DIR, 'valid_candidates.sdf'))
print(f"  ✓ Staged valid_candidates.sdf")

# 2. valid_trajectories/
staged_traj = os.path.join(STAGING_DIR, 'valid_trajectories')
if os.path.isdir(VALID_TRAJ_DIR):
    shutil.copytree(VALID_TRAJ_DIR, staged_traj)
    n_traj = len(glob.glob(os.path.join(staged_traj, '*.xyz')))
    print(f"  ✓ Staged valid_trajectories/ ({n_traj} files)")
else:
    os.makedirs(staged_traj)
    print(f"  ⚠ No valid trajectories found — staging empty directory")

# 3. ensemble_receptors/
ens_dir = os.path.join(STAGING_DIR, "ensemble_receptors")
os.makedirs(ens_dir, exist_ok=True)
shutil.copy2(POCKET_PDB, ens_dir)

from Bio.PDB.MMCIFParser import MMCIFParser
from Bio.PDB.PDBIO import PDBIO

cif_parser = MMCIFParser(QUIET=True)
io = PDBIO()

# 1. Find all raw CIFs generated by ConforMix (ignoring the .xtc trajectory)
cif_files = glob.glob("/content/conformix_outputs/**/*.cif", recursive=True)

print(f"Found {len(cif_files)} ConforMix CIF files. Converting to PDB...")

# 2. Convert and stage each variant
for i, cif_path in enumerate(cif_files):
    try:
        struct = cif_parser.get_structure(f"variant_{i}", cif_path)
        io.set_structure(struct)
        
        out_pdb = os.path.join(STAGING_DIR, 'ensemble_receptors', f"conformix_var_{i}.pdb")
        io.save(out_pdb)
        print(f"  ✓ Converted: conformix_var_{i}.pdb")
    except Exception as e:
        print(f"  ⚠ Failed to convert {cif_path}: {e}")

print(f"  ✓ Staged ensemble_receptors/ ({len(cif_files)+1} files)")
# 4. metadata.json — pocket center, radius, and PDB ID for downstream docking
metadata = {
    "pdb_id": PDB_ID,
    "pocket_center": list(POCKET_CENTER),
    "pocket_radius": POCKET_RADIUS,
    "n_samples_generated": N_SAMPLES,
    "n_valid_candidates": len([m for m in open(VALID_SDF).read().split('$$$$') if m.strip()]) if os.path.isfile(VALID_SDF) else 0,
}
meta_path = os.path.join(STAGING_DIR, 'metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"  ✓ Staged metadata.json")
print(f"    pdb_id         : {metadata['pdb_id']}")
print(f"    pocket_center  : {metadata['pocket_center']}")
print(f"    pocket_radius  : {metadata['pocket_radius']}")
print(f"    valid_candidates: {metadata['n_valid_candidates']}")

# ── Create the zip archive ────────────────────────────────────────────────────
zip_base = PAYLOAD_ZIP.replace('.zip', '')
archive_path = shutil.make_archive(zip_base, 'zip', STAGING_DIR)
size_mb = os.path.getsize(archive_path) / 1_000_000

print(f"\n{'═' * 60}")
print(f"✅ Payload ready: {archive_path} ({size_mb:.2f} MB)")
print(f"{'═' * 60}")

# List contents
print(f"\nContents:")
for root, dirs, files in os.walk(STAGING_DIR):
    level = root.replace(STAGING_DIR, '').count(os.sep)
    indent = '  ' * (level + 1)
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = '  ' * (level + 2)
    for file in sorted(files):
        fsize = os.path.getsize(os.path.join(root, file)) / 1_000
        print(f"{sub_indent}{file} ({fsize:.1f} KB)")

# ── Cleanup staging directory ─────────────────────────────────────────────────
shutil.rmtree(STAGING_DIR)

# ── Upload to transfer.sh for download ────────────────────────────────────────
import subprocess
print(f"\n⏳ Uploading {size_mb:.1f} MB to transfer.sh…")
result = subprocess.run(
    ['curl', '--upload-file', archive_path, 'https://transfer.sh/ensemble_payload.zip'],
    capture_output=True, text=True, timeout=120
)
if result.returncode == 0 and result.stdout.strip().startswith('http'):
    download_url = result.stdout.strip()
    print(f"\n{'═' * 60}")
    print(f"📥 DOWNLOAD LINK (valid ~14 days):")
    print(f"\n   {download_url}")
    print(f"\n{'═' * 60}")
    print(f"\nTo download locally, run:")
    print(f'   curl -L -o ensemble_payload.zip "{download_url}"')
else:
    print(f"\n⚠ transfer.sh upload failed. File is still at: {archive_path}")
    if result.stderr:
        print(f"  Error: {result.stderr[:200]}")
    print(f"\nAlternative: mount Google Drive and copy manually:")
    print(f"  from google.colab import drive")
    print(f"  drive.mount('/content/drive')")
    print(f"  !cp {archive_path} /content/drive/MyDrive/")


  ✓ Staged valid_candidates.sdf
  ✓ Staged valid_trajectories/ (8 files)
Found 6 ConforMix CIF files. Converting to PDB...
  ✓ Converted: conformix_var_0.pdb
  ✓ Converted: conformix_var_1.pdb
  ✓ Converted: conformix_var_2.pdb
  ✓ Converted: conformix_var_3.pdb
  ✓ Converted: conformix_var_4.pdb
  ✓ Converted: conformix_var_5.pdb
  ✓ Staged ensemble_receptors/ (7 files)
  ✓ Staged metadata.json
    pdb_id         : 3PTB
    pocket_center  : [17.0, 14.0, 4.0]
    pocket_radius  : 10.0
    valid_candidates: 8

════════════════════════════════════════════════════════════
✅ Payload ready: /content/ensemble_payload.zip (0.72 MB)
════════════════════════════════════════════════════════════

Contents:
  ensemble_payload_staging/
    metadata.json (0.2 KB)
    valid_candidates.sdf (8.1 KB)
    valid_trajectories/
      mol_0001.xyz (211.4 KB)
      mol_0002.xyz (247.4 KB)
      mol_0003.xyz (265.4 KB)
      mol_0004.xyz (229.4 KB)
      mol_0005.xyz (211.4 KB)
      mol_0006.xyz (283.4 KB)
  

## Temporary: Google Drive connectivity for payload download

In [16]:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/ensemble_payload.zip /content/drive/MyDrive/
print("✅ Copied to Google Drive root")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Copied to Google Drive root


In [17]:
from google.colab import runtime
runtime.unassign()